In [1]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/adult.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
df.head()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,workclass,education,marital-status,occupation,relationship,race,sex,native-country,class
0,19,134974.0,10,0.0,0.0,20,nan,Some-college,Never-married,nan,Own-child,White,Female,United-States,<=50K
1,41,195096.0,13,0.0,0.0,50,Self-emp-inc,Bachelors,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States,<=50K
2,31,152109.0,9,0.0,0.0,50,Private,HS-grad,Never-married,Exec-managerial,Not-in-family,White,Male,United-States,<=50K
3,40,202872.0,12,0.0,0.0,45,Private,Assoc-acdm,Never-married,Adm-clerical,Own-child,White,Female,United-States,<=50K
4,35,98989.0,5,0.0,0.0,38,nan,9th,Divorced,nan,Own-child,Amer-Indian-Eskimo,Male,United-States,<=50K


In [3]:
from preprocessor import clean_dataset
df = clean_dataset(df,['fnlwgt', 'education-num'])
df.to_csv("cleaned.csv", index=True)

In [5]:
df['class'] = df['class'].map({'<=50K' : 0, '>50K' : 1})
y = df["class"]
X = df.drop(["class"], axis=1)

In [7]:
column_types = {
        "sufficient_resources_nominal_columns" : ["workclass", "marital-status", "occupation", "relationship", "race", "sex" ],
        "insufficient_resources_nominal_columns": None,
        "binary": None,
        "ordinal": None,
        "inclusion": ["education"],
        "similarity":None,
    }
ANOC_threshold = 100

In [13]:
import time
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from preprocessor import clean_dataset
from meta_encoder import MetaEncoder
from sklearn.base import clone

# models
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

#from approach import encode_with_approach

model_types = {
    "LogReg": "ATI",
    "MLP": "ATI",
    "LinearSVM": "ATI",
    "CART": "tree",
    "RF": "tree",
    "CatBoost": "tree",
    "XGBoost": "tree",
    "LightGBM": "tree"
}
target_encoders = ['Leave-one-out', 'K-fold', 'CatBoost', 'CV-GLMM', 'WOE']

nominal_cols = column_types["sufficient_resources_nominal_columns"].copy()
if column_types["insufficient_resources_nominal_columns"]:
    nominal_cols.extend(column_types["insufficient_resources_nominal_columns"])

max_categories = df[nominal_cols].nunique().max()
ANOC = df.shape[0] / max_categories
# ANOC = 101

encode_with_target: bool = False

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():

    # initializing guide parameters
    if model_types[model_name] == 'ATI':
        if ANOC > ANOC_threshold:
            encode_with_target = False
        else:
            encode_with_target = True
    else:  # 'tree-based'
        if ANOC > ANOC_threshold:
            encode_with_target = True
        else:
            encode_with_target = False

    target_encoders_to_apply = []

    if encode_with_target:
        target_encoders_to_apply = target_encoders
    else: # it just a mark that we use target-agnostic encoders
        target_encoders_to_apply = ["OneHot"]

    for nominal_encoder in target_encoders_to_apply:

        # creating parameters for the meta-encoder
        encode_columns_as = {}
        for type, columns in column_types.items():
            if type == "binary":
                encode_columns_as["Binary"] = columns
                continue
            if type == "similarity":
                encode_columns_as["Similarity"] = columns
                continue
            if type == "ordinal":
                encode_columns_as["Ordinal"] = columns
                continue
            if type == "inclusion":
                encode_columns_as["RankHot"] = columns
                continue

        encoder_name: str
        if not encode_with_target:
            if column_types["sufficient_resources_nominal_columns"] :
                encode_columns_as["OneHot"] = column_types["sufficient_resources_nominal_columns"]
                encoder_name = "OneHot"
            if column_types["insufficient_resources_nominal_columns"] :
                encode_columns_as["Hash"] = column_types["insufficient_resources_nominal_columns"]
                if encoder_name != '':
                    encoder_name = encoder_name + "_Hash"
        else:
            encode_columns_as[nominal_encoder] = nominal_cols
            encoder_name = nominal_encoder

        encoder = MetaEncoder(encode_columns_as=encode_columns_as)

        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            X_train_encoded = encoder.fit_transform(X_train, y_train)
            X_test_encoded = encoder.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Approach",
                    "dataset": "Adults",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Approach, Adults, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_adults_approach.csv", index=True)
print("Done!")


Approach, Adults, LogReg, OneHot, 1, 1, 0.8850957753447485, 0.6992449665259337, 0.8337598525949432, 0.6079188797682279, 0.6978935698447893, 0.5384944396920445, 2.659712314605713
Approach, Adults, LogReg, OneHot, 1, 2, 0.883039676457685, 0.6815431343319023, 0.8366260620329614, 0.6225165562913907, 0.6962962962962963, 0.562874251497006, 2.765413761138916
Approach, Adults, LogReg, OneHot, 1, 3, 0.8893747163679061, 0.6936812774769118, 0.8370188370188371, 0.6211327939076631, 0.6997319034852547, 0.558408215661104, 2.7486746311187744
Approach, Adults, LogReg, OneHot, 1, 4, 0.881536062454945, 0.684006825849, 0.8289312039312039, 0.595105403440756, 0.6860335195530727, 0.5254599914420197, 3.0289885997772217
Approach, Adults, LogReg, OneHot, 1, 5, 0.885477616436067, 0.691561883013631, 0.8347665847665847, 0.6188001889466226, 0.6905640484976279, 0.5605477107402653, 2.765685558319092
Approach, Adults, LogReg, OneHot, 2, 1, 0.885818679268719, 0.7026758741491276, 0.8368307912785341, 0.6192068800764453, 